In [0]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


df = pd.read_csv("orders_sample_ic410.csv")

df["revenue"] = df["quantity"] * df["price_each"]

sort_values = df.sort_values(by=["customer_id", "revenue", "order_id"], ascending=[True, False, True]).groupby("customer_id").head(1)

print(sort_values)

df_running = df.sort_values(by= ["customer_id", "placed_at"], ascending=[True,True])
df_running["running_revenue"] = df_running.groupby("customer_id")["revenue"].cumsum()
print(df_running)

spark = SparkSession.builder \
    .appName("IC4.10") \
    .getOrCreate()

df = spark.read.csv(
    "orders_sample_ic410.csv",
    header=True,
    inferSchema=True
)

revenue = F.col("quantity") * F.col("price_each")
df = df.withColumn("revenue", revenue)

top_window = Window.partitionBy("customer_id").orderBy(
    F.col("revenue").desc(),
    F.col("order_id").asc()
    )
top_per_customer = df.withColumn("row_num", F.row_number().over(top_window)).where(F.col("row_num") == 1)

window_running  = Window.partitionBy("customer_id").orderBy("placed_at")
df_running = df.withColumn("running_revenue", F.sum("revenue").over(window_running))

